# Data Preparation

## Create Train Files

We first generate the list of files we want to use to test.

In [ ]:
import pickle
import os
import random

#### Train Large

In [ ]:
# create list of all pairs of recordings for training and testing
train_pairs = []

train_dir = "Chopin_Mazurkas/annotations_beat/Chopin_Op017No4"

# create list of all pairs of recordings in the training directory
movement_path = os.path.join(train_dir)
train_list = [f.split('.')[0] for f in os.listdir(movement_path) if f.endswith('.beat')]

for i in range(len(train_list)):
    for j in range(i + 1, len(train_list)):
        train_pairs.append((train_list[i], train_list[j]))
        
with open('cfg/mazurkas.train_large.pkl', 'wb') as f:
    pickle.dump(train_list, f)
        
with open('cfg/mazurkas.train_pairs_large.pkl', 'wb') as f:
    pickle.dump(train_pairs, f)
    
len(train_pairs)

#### Train Small

In [ ]:
# sample 10 pieces from train_
random.seed(42)
train_small = random.sample(train_list, 10)
train_pairs_small = []
for i in range(len(train_small)):
    for j in range(i + 1, len(train_small)):
        train_pairs_small.append((train_small[i], train_small[j]))
with open('cfg/mazurkas.train.pkl', 'wb') as f:
    pickle.dump(train_small, f)
with open('cfg/mazurkas.train_pairs.pkl', 'wb') as f:
    pickle.dump(train_pairs_small, f)

## Alignment Scenarios

In [ ]:
def generateScenarios(outdir, pairs_list):
    """
    Generate all possible alignment scenarios for a given output directory.
    
    Args:
        outdir: output directory
        pairs_list: list of pairs of recordings to process. Each pair is a tuple of (query, reference)
    """
    # create output directory
    if os.path.exists(outdir):
        print(f"Directory {outdir}/ already exists. Deleting and regenerating...")
        import shutil
        shutil.rmtree(outdir)
    os.mkdir(outdir)
    
    for i, (query, ref) in enumerate(pairs_list):
        scenario_id  = f"s{i+1}"
        # create scenario directory
        scenario_dir = os.path.join(outdir, scenario_id)
        os.makedirs(scenario_dir, exist_ok=True)
        
        # generate symbolic links for query and reference audio files
        cwd = os.getcwd()
        query_path = os.path.join(cwd, "Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4", f"{query}.wav")
        ref_path = os.path.join(cwd, "Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4", f"{ref}.wav")
        query_link = os.path.join(scenario_dir, "query.wav")
        ref_link = os.path.join(scenario_dir, "ref.wav")
        os.symlink(query_path, query_link)
        os.symlink(ref_path, ref_link)
        
        # generate symbolic links for annotation files
        query_annot_path = os.path.join(cwd, "Chopin_Mazurkas/annotations_beat/Chopin_Op017No4", f"{query}.beat")
        ref_annot_path = os.path.join(cwd, "Chopin_Mazurkas/annotations_beat/Chopin_Op017No4", f"{ref}.beat")
        query_annot_link = os.path.join(scenario_dir, "query.beats")
        ref_annot_link = os.path.join(scenario_dir, "ref.beats")
        os.symlink(query_annot_path, query_annot_link)
        os.symlink(ref_annot_path, ref_annot_link)
        
        # generate text file storing the query and reference
        text = f"{query} {ref}\n"
        with open(os.path.join(scenario_dir, "pair.txt"), "w") as f:
            f.write(text)
        

In [ ]:
SCENARIOS_DIR = "scenarios"
generateScenarios(SCENARIOS_DIR, train_pairs_small)

## Compute Features

We first want to compute and store the chroma stft features.

In [ ]:
import pickle
import os
import librosa as lb
import numpy as np
from tqdm.notebook import tqdm
import utils.constants as constants

TRAIN_FILE = "cfg/mazurkas.train_large.pkl" # select train small or train large. "train.pkl" for train small and "train_large".pkl for full training benchmark
AUDIO_ROOT = "Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4"
FEAT_DIR = "features"
CHROMA_STFT_DIR = f"{FEAT_DIR}/chroma_stft_norm2"
MATCH_DIR = f"{FEAT_DIR}/match"

In [ ]:
# make feature directories
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(CHROMA_STFT_DIR, exist_ok=True)
os.makedirs(MATCH_DIR, exist_ok=True)

In [ ]:
# load piece ids
with open(TRAIN_FILE, "rb") as f:
    piece_ids = pickle.load(f)
    
# compute STFT features
for piece_id in tqdm(piece_ids, desc="Computing chroma_stft"):
    chroma_stft_path = f"{CHROMA_STFT_DIR}/{piece_id}.npy"
    if os.path.exists(chroma_stft_path):
        print(f"Features for {piece_id} already exists at {chroma_stft_path}.")
        continue
    audio_path = f"{AUDIO_ROOT}/{piece_id}.wav"
    y, sr = lb.load(audio_path)
    chroma_stft_feat = lb.feature.chroma_stft(y=y, sr=sr, hop_length=constants.DEFAULT_HOP_LENGTH, center=False, norm=2)
    np.save(chroma_stft_path, chroma_stft_feat)

In [ ]:
from utils.match_features import extract_match_features

# load piece ids
with open(TRAIN_FILE, "rb") as f:
    piece_ids = pickle.load(f)
    
# compute match features only if needed
for piece_id in tqdm(piece_ids, desc="Computing match features"):
    audio_path = f"{AUDIO_ROOT}/{piece_id}.wav"
    y, sr = lb.load(audio_path)
    match_feat = extract_match_features(audio_path)
    match_feat_path = f"{MATCH_DIR}/{piece_id}.npy"
    np.save(match_feat_path, match_feat)